# Exercise 38: Exchange Seats

**Source:** [LeetCode 626](https://leetcode.com/problems/exchange-seats/description/?envType=study-plan-v2&envId=top-sql-50)

Table: `Seat`
```
+-------------+---------+
| Column Name | Type    |
+-------------+---------+
| id          | int     |
| student     | varchar |
+-------------+---------+
```
- `id` is the primary key which increases always by 1,
- and `name` is the student with the `id`.

**Task.**
Swap every two consecutive students.
If the total number is odd, the last student's id is not swapped.
The result must be returned in **ascending order** by `id`.

Connect to the database `postgres` using the username `postgres`.

In [1]:
%load_ext sql
%sql postgresql+psycopg://postgres@localhost:5432/postgres

The 'toml' package isn't installed. To load settings from pyproject.toml or ~/.jupysql/config, install with: pip install toml

Connecting to 'postgresql+psycopg://postgres@localhost:5432/postgres'

### Create new database for the exercise

Make sure first that a database with the same name is removed in case it already exists.

In [2]:
%sql DROP DATABASE IF EXISTS exercise_38 WITH (FORCE);
%sql CREATE DATABASE exercise_38;

Running query in 'postgresql+psycopg://postgres@localhost:5432/postgres'

Running query in 'postgresql+psycopg://postgres@localhost:5432/postgres'

++
||
++
++

Connect to the database.

*Caution.* No semicolon at the end otherwise it belongs to the name of the database.

In [3]:
%sql postgresql+psycopg://postgres@localhost:5432/exercise_38 --alias exercise_38

Connecting and switching to connection 'exercise_38'

#### Fill the database with content

In [4]:
%%sql

DROP TABLE IF EXISTS Seat;

CREATE TABLE Seat (
    id SERIAL PRIMARY KEY,
    student VARCHAR NOT NULL
);

INSERT INTO Seat (student)
VALUES
('Abbot'),
('Doris'),
('Emerson'),
('Green'),
('Jeames');

SELECT * FROM Seat

Running query in 'exercise_38'

5 rows affected.

5 rows affected.

id,student
1,Abbot
2,Doris
3,Emerson
4,Green
5,Jeames


#### Solve the exercise

**Task.**
Swap every two consecutive students.
If the total number is odd, the last student's id is not swapped.
The result must be returned in **ascending order** by `id`.

In [5]:
%%sql
SELECT P.id,
    CASE WHEN (P.id % 2 = 1 AND T.id IS NOT NULL) THEN T.student
         WHEN P.id % 2 = 0 THEN S.student
         ELSE P.student
    END
FROM Seat P LEFT JOIN Seat S ON P.id = S.id + 1 LEFT JOIN Seat T ON P.id = T.id-1;

Running query in 'exercise_38'

5 rows affected.

id,student
1,Doris
2,Abbot
3,Green
4,Emerson
5,Jeames


#### Explanation

- Doing a `LEFT JOIN` with the primary table `Seat P` and a second copy of `Seat S` on `P.id = S.id + 1` and then a `LEFT JOIN` with the primary table `Seat P` and a third copy of `Seat T` on `P.id = T.id - 1`, one needs to select `P.id` and the name based on whether `P.id` is even or odd.
- Do a case distinction with `CASE WHEN ... THEN ...` ending with `ELSE ... END`.
- If `P.id` is odd (and `T.id` not `NULL` which happens for the last row with an odd number of rows), one needs to choose `T.student` with `P.id`.
- If `P.id` is even, one needs to choose `S.student` with `P.id`.
- There remains the case where the total number of rows is odd, handled in `ELSE`-clause. This does mean that no change is wanted.

**Remark.**
It is possible to avoid the big join, but needing the order-clause to keep the ordering by `id`.

In [7]:
%%sql
SELECT
    CASE
        WHEN id % 2 = 1 AND id = (SELECT COUNT(*) FROM Seat) THEN id
        WHEN id % 2 = 1 THEN id + 1
        ELSE id - 1
    END AS id,
    student
FROM Seat
ORDER BY id;

Running query in 'exercise_38'

5 rows affected.

id,student
1,Doris
2,Abbot
3,Green
4,Emerson
5,Jeames


#### Close connection

In [8]:
%sql --close exercise_38
%sql postgresql+psycopg://postgres@localhost:5432/postgres
%sql DROP DATABASE IF EXISTS exercise_38;

Switching to connection 'postgresql+psycopg://postgres@localhost:5432/postgres'

Running query in 'postgresql+psycopg://postgres@localhost:5432/postgres'

++
||
++
++

**Well done!** (approved by Leet)